In [4]:
from pathlib import Path
import shutil

# =========================
# 1. 基本設定
# =========================

# 用 list 放所有要處理的資料夾
TARGET_ROOTS = [
    Path(r"data/右45-20260416T120504Z-3-001"),
    Path(r"data/左45-20260416T120505Z-3-001"),
    Path(r"data/正右-20260416T120506Z-3-001"),
    Path(r"data/正右-20260416T120507Z-3-001"),
    Path(r"data/正左-20260416T120508Z-3-001"),
    Path(r"data/正左-20260416T120509Z-3-001"),
    Path(r"data/正面-20260416T120510Z-3-001"),
    Path(r"data/正面-20260416T120510Z-3-002"),
    Path(r"data/正面-20260416T120510Z-3-003"),
    Path(r"data/正面-20260416T120510Z-3-004"),
    Path(r"data/正面-20260416T120510Z-3-005"),
    Path(r"data/正面-20260416T120510Z-3-006"),
    Path(r"data/正面-20260416T120511Z-3-001"),
    Path(r"data/正面-20260416T120511Z-3-002"),
    Path(r"data/正面-20260416T120511Z-3-003"),
    Path(r"data/正面-20260416T120511Z-3-004"),
    Path(r"data/正面-20260416T120511Z-3-005"),
    Path(r"data/正面-20260416T120511Z-3-006"),
]

# 整理後輸出位置
OUTPUT_ROOT = Path(r"data/new")

# True = 移動檔案（原位置會消失）
# False = 複製檔案（原位置保留）
MOVE_FILES = True

# True = 只預覽，不真的執行
DRY_RUN = False

# 支援的影片副檔名
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".flv", ".mpeg", ".mpg", ".m4v"}

# 合法動作類別
ACTION_NAMES = {
    "elbow_flexion_left",
    "elbow_flexion_right",
    "shoulder_abduction_left",
    "shoulder_abduction_right",
    "shoulder_backward_left",
    "shoulder_backward_right",
    "shoulder_flexion_left",
    "shoulder_flexion_right",
    "shoulder_forward_elevation",
    "side_tap_left",
    "side_tap_right",
}


# =========================
# 2. 工具函式
# =========================

def is_video_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in VIDEO_EXTS


def find_action_folder(video_path: Path):
    """
    從影片路徑往上找對應的動作資料夾名稱
    例如：
    data/右45-20260416T120504Z-3-001/右45/elbow_flexion_left/xxx.mp4
    -> elbow_flexion_left
    """
    for parent in video_path.parents:
        if parent.name in ACTION_NAMES:
            return parent.name
    return None


def get_start_index(folder: Path, action_name: str) -> int:
    """
    若目標資料夾已經有舊檔，接續編號
    例如已有 shoulder_flexion_left_001.mp4 ~ 005.mp4
    就從 006 開始
    """
    if not folder.exists():
        return 1

    max_idx = 0
    pattern = f"{action_name}_*"

    for f in folder.glob(pattern):
        if not f.is_file():
            continue
        stem = f.stem  # 不含副檔名
        parts = stem.split("_")
        if not parts:
            continue
        last_part = parts[-1]
        if last_part.isdigit():
            max_idx = max(max_idx, int(last_part))

    return max_idx + 1


# =========================
# 3. 主程式
# =========================

def main():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    action_to_files = {}

    # 只掃描 TARGET_ROOTS 裡面指定的資料夾
    for target_root in TARGET_ROOTS:
        if not target_root.exists():
            print(f"[警告] 找不到資料夾，略過：{target_root}")
            continue

        print(f"\n=== 掃描資料夾：{target_root} ===")

        for path in target_root.rglob("*"):
            if not is_video_file(path):
                continue

            action_name = find_action_folder(path)
            if action_name is None:
                print(f"[略過] 無法判斷動作類別：{path}")
                continue

            action_to_files.setdefault(action_name, []).append(path)

    if not action_to_files:
        print("\n沒有找到任何可處理的影片。")
        return

    total_count = 0

    for action_name in sorted(action_to_files.keys()):
        files = sorted(action_to_files[action_name])
        target_dir = OUTPUT_ROOT / action_name
        target_dir.mkdir(parents=True, exist_ok=True)

        start_index = get_start_index(target_dir, action_name)

        print(f"\n[處理類別] {action_name}")
        print(f"輸出資料夾：{target_dir}")

        for offset, src_file in enumerate(files):
            idx = start_index + offset
            ext = src_file.suffix.lower()
            new_name = f"{action_name}_{idx:03d}{ext}"
            dst_file = target_dir / new_name

            print(f"{src_file} -> {dst_file}")

            if not DRY_RUN:
                if MOVE_FILES:
                    shutil.move(str(src_file), str(dst_file))
                else:
                    shutil.copy2(str(src_file), str(dst_file))

            total_count += 1

    print("\n======================")
    print(f"完成，共處理 {total_count} 個影片")
    print(f"輸出資料夾：{OUTPUT_ROOT}")
    print("======================")


if __name__ == "__main__":
    main()


=== 掃描資料夾：data\右45-20260416T120504Z-3-001 ===

=== 掃描資料夾：data\左45-20260416T120505Z-3-001 ===

=== 掃描資料夾：data\正右-20260416T120506Z-3-001 ===

=== 掃描資料夾：data\正右-20260416T120507Z-3-001 ===

=== 掃描資料夾：data\正左-20260416T120508Z-3-001 ===

=== 掃描資料夾：data\正左-20260416T120509Z-3-001 ===

=== 掃描資料夾：data\正面-20260416T120510Z-3-001 ===

=== 掃描資料夾：data\正面-20260416T120510Z-3-002 ===

=== 掃描資料夾：data\正面-20260416T120510Z-3-003 ===

=== 掃描資料夾：data\正面-20260416T120510Z-3-004 ===

=== 掃描資料夾：data\正面-20260416T120510Z-3-005 ===

=== 掃描資料夾：data\正面-20260416T120510Z-3-006 ===

=== 掃描資料夾：data\正面-20260416T120511Z-3-001 ===

=== 掃描資料夾：data\正面-20260416T120511Z-3-002 ===

=== 掃描資料夾：data\正面-20260416T120511Z-3-003 ===

=== 掃描資料夾：data\正面-20260416T120511Z-3-004 ===

=== 掃描資料夾：data\正面-20260416T120511Z-3-005 ===

=== 掃描資料夾：data\正面-20260416T120511Z-3-006 ===

[處理類別] elbow_flexion_left
輸出資料夾：data\new\elbow_flexion_left
data\右45-20260416T120504Z-3-001\右45\elbow_flexion_left\elbow_flexion_left_right45_01.mp4 -> data\new\elbow_fl

In [1]:
# =========================
# Test Set 影片重新命名
# 把 data/test/<action>/ 內所有影片改名為 <action>_001.mp4 ... 等格式
# =========================
from pathlib import Path
import shutil
import uuid

# 1) 基本設定
TEST_ROOT = Path(r"data/test")

# True = 直接 rename（原檔名消失）
# False = 複製到新檔名，原檔保留
RENAME_IN_PLACE = True

# True = 只預覽，不真的執行
DRY_RUN = False

# 是否將編號歸零重編（True = 從 001 重新開始；False = 接續既有編號）
RESET_INDEX = True

# 支援的影片副檔名
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".flv", ".mpeg", ".mpg", ".m4v"}

# 訓練腳本使用的 9 個合法類別（與 yolo11_detect_person_trf.py 的 LABELS 完全一致）
VALID_LABELS = {
    "shoulder_abduction_left", "shoulder_abduction_right",
    "shoulder_flexion_left",   "shoulder_flexion_right",
    "shoulder_backward_left",  "shoulder_backward_right",
    "elbow_flexion_left",      "elbow_flexion_right",
    "shoulder_forward_elevation",
}


# 2) 取得下一個可用編號（避免覆蓋已重新命名過的檔案）
def get_start_index(folder: Path, action_name: str) -> int:
    if RESET_INDEX or not folder.exists():
        return 1
    max_idx = 0
    for f in folder.glob(f"{action_name}_*"):
        if not f.is_file():
            continue
        last = f.stem.split("_")[-1]
        if last.isdigit():
            max_idx = max(max_idx, int(last))
    return max_idx + 1


# 3) 主程式
def rename_test_set():
    if not TEST_ROOT.exists():
        print(f"[錯誤] 找不到 test 根目錄：{TEST_ROOT}")
        return

    print(f"=== Test set 根目錄：{TEST_ROOT.resolve()} ===")
    print(f"模式：{'DRY-RUN（不實際變動）' if DRY_RUN else '實際執行'}"
          f" / {'RENAME' if RENAME_IN_PLACE else 'COPY'}"
          f" / {'RESET' if RESET_INDEX else 'APPEND'} 編號\n")

    total = 0
    skipped_dirs = []

    # 遍歷每個類別資料夾
    for action_dir in sorted(TEST_ROOT.iterdir()):
        if not action_dir.is_dir():
            continue
        action_name = action_dir.name
        if action_name not in VALID_LABELS:
            skipped_dirs.append(action_name)
            continue

        # 收集影片檔
        all_videos = sorted(
            p for p in action_dir.iterdir()
            if p.is_file() and p.suffix.lower() in VIDEO_EXTS
        )

        # 第一階段：先全部改成暫存名，避免「A→B 但 B 已存在」的衝突
        temp_pairs = []
        for p in all_videos:
            tmp = p.with_name(f".__tmp_{uuid.uuid4().hex[:8]}_{p.name}")
            if not DRY_RUN:
                p.rename(tmp) if RENAME_IN_PLACE else shutil.copy2(str(p), str(tmp))
            temp_pairs.append((tmp if not DRY_RUN else p, p.suffix.lower()))

        # 第二階段：依序給予最終名稱
        start = get_start_index(action_dir, action_name)
        print(f"[{action_name}] 共 {len(temp_pairs)} 支，起始編號 = {start:03d}")
        for offset, (tmp_path, ext) in enumerate(temp_pairs):
            new_name = f"{action_name}_{start + offset:03d}{ext}"
            dst = action_dir / new_name
            print(f"  {tmp_path.name}  ->  {new_name}")
            if not DRY_RUN:
                if RENAME_IN_PLACE:
                    tmp_path.rename(dst)
                else:
                    shutil.move(str(tmp_path), str(dst))
            total += 1

    print(f"\n======================")
    print(f"完成：共處理 {total} 支影片")
    if skipped_dirs:
        print(f"略過非法類別資料夾：{skipped_dirs}")
    print(f"======================")


# 執行
rename_test_set()


=== Test set 根目錄：D:\UserData\07大學教材\專題\Code\data\test ===
模式：實際執行 / RENAME / RESET 編號

[elbow_flexion_left] 共 65 支，起始編號 = 001
  .__tmp_61eb2993_IMG_2241.MOV  ->  elbow_flexion_left_001.mov
  .__tmp_de259153_IMG_2242.MOV  ->  elbow_flexion_left_002.mov
  .__tmp_9a5bcedf_IMG_2271.MOV  ->  elbow_flexion_left_003.mov
  .__tmp_f153a76f_IMG_2272.MOV  ->  elbow_flexion_left_004.mov
  .__tmp_f0f2e2f8_IMG_2273.MOV  ->  elbow_flexion_left_005.mov
  .__tmp_11683bb3_IMG_2296.MOV  ->  elbow_flexion_left_006.mov
  .__tmp_4cca71cd_IMG_2297.MOV  ->  elbow_flexion_left_007.mov
  .__tmp_4183176a_IMG_2298.MOV  ->  elbow_flexion_left_008.mov
  .__tmp_6ed5c440_IMG_2320.MOV  ->  elbow_flexion_left_009.mov
  .__tmp_92db61c8_IMG_2321.MOV  ->  elbow_flexion_left_010.mov
  .__tmp_e1264f1e_IMG_2322.MOV  ->  elbow_flexion_left_011.mov
  .__tmp_1cdfa3cc_IMG_2339.MOV  ->  elbow_flexion_left_012.mov
  .__tmp_31bcc340_IMG_2340.MOV  ->  elbow_flexion_left_013.mov
  .__tmp_ecf8e155_IMG_2341.MOV  ->  elbow_flexion_left_

In [1]:
from pathlib import Path

# =========================
# 1. 基本設定
# =========================
DATA_ROOT = Path("data")

# 只處理這些受試者資料夾
SUBJECTS = [f"P{i:03d}" for i in range(1, 7)]  # P001 ~ P006

# 支援的影片格式
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".wmv", ".flv", ".mpeg", ".mpg", ".m4v"}

# True = 只顯示將怎麼改名，不真的改
DRY_RUN = False


def is_video_file(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in VIDEO_EXTS


def main():
    if not DATA_ROOT.exists():
        print(f"找不到資料夾：{DATA_ROOT}")
        return

    total_count = 0

    for subject in SUBJECTS:
        subject_dir = DATA_ROOT / subject

        if not subject_dir.exists():
            print(f"[略過] 找不到受試者資料夾：{subject_dir}")
            continue

        print(f"\n=== 處理受試者：{subject} ===")

        # 每個 subject 底下的第一層資料夾視為動作資料夾
        action_dirs = [p for p in subject_dir.iterdir() if p.is_dir()]

        for action_dir in sorted(action_dirs):
            action_name = action_dir.name

            video_files = [p for p in action_dir.iterdir() if is_video_file(p)]
            video_files = sorted(video_files)

            if not video_files:
                print(f"[略過] {action_dir} 沒有影片")
                continue

            print(f"\n[動作] {action_name}")

            for idx, old_path in enumerate(video_files, start=1):
                # 保留原始副檔名
                ext = old_path.suffix
                new_name = f"{subject}_{action_name}_{idx:03d}{ext}"
                new_path = action_dir / new_name

                # 若原本名稱已經正確，略過
                if old_path.name == new_name:
                    print(f"[略過] 已符合格式：{old_path.name}")
                    continue

                print(f"{old_path.name} -> {new_name}")

                if not DRY_RUN:
                    if new_path.exists():
                        print(f"[警告] 目標檔案已存在，略過：{new_path}")
                        continue
                    old_path.rename(new_path)

                total_count += 1

    print("\n======================")
    print(f"完成，共處理 {total_count} 個影片檔")
    print("======================")


if __name__ == "__main__":
    main()


=== 處理受試者：P001 ===

[動作] elbow_flexion_left
[略過] 已符合格式：P001_elbow_flexion_left_001.mp4
[略過] 已符合格式：P001_elbow_flexion_left_002.mp4
[略過] 已符合格式：P001_elbow_flexion_left_003.mp4
[略過] 已符合格式：P001_elbow_flexion_left_004.mp4
[略過] 已符合格式：P001_elbow_flexion_left_005.mp4
[略過] 已符合格式：P001_elbow_flexion_left_006.mp4
[略過] 已符合格式：P001_elbow_flexion_left_007.mp4
[略過] 已符合格式：P001_elbow_flexion_left_008.mp4
[略過] 已符合格式：P001_elbow_flexion_left_009.mp4
[略過] 已符合格式：P001_elbow_flexion_left_010.mp4
[略過] 已符合格式：P001_elbow_flexion_left_011.mp4
[略過] 已符合格式：P001_elbow_flexion_left_012.mp4
[略過] 已符合格式：P001_elbow_flexion_left_013.mp4
[略過] 已符合格式：P001_elbow_flexion_left_014.mp4
[略過] 已符合格式：P001_elbow_flexion_left_015.mp4
[略過] 已符合格式：P001_elbow_flexion_left_016.mp4
[略過] 已符合格式：P001_elbow_flexion_left_017.mp4
[略過] 已符合格式：P001_elbow_flexion_left_018.mp4
[略過] 已符合格式：P001_elbow_flexion_left_019.mp4
[略過] 已符合格式：P001_elbow_flexion_left_020.mp4
[略過] 已符合格式：P001_elbow_flexion_left_021.mp4
[略過] 已符合格式：P001_elbow_flexion_left_022.mp4
[略過] 已符合格

In [4]:
import hashlib, os

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

# 掃整個資料集找完全相同的檔案
seen = {}
for root, _, files in os.walk("data"):
    for fname in files:
        if not fname.lower().endswith((".mp4", ".mov", ".avi", ".mkv")):
            continue
        p = os.path.join(root, fname)
        h = md5(p)
        if h in seen:
            print(f"DUPLICATE: {p}  ←→  {seen[h]}")
        else:
            seen[h] = p

In [ ]:
"""
deduplicate_dataset.py
======================
掃描 data/ 目錄，找出 MD5 完全相同的影片（真正的重複檔案），
保留路徑字母順序最靠前的一份，刪除其餘重複檔案，
並同時刪除對應的特徵快取（data/video_features_v1/*.npz）。

用法：
  # 【強烈建議先跑這步】預覽，不刪任何檔案
  python deduplicate_dataset.py --dry-run

  # 確認無誤後，實際執行刪除
  python deduplicate_dataset.py

  # 自訂路徑
  python deduplicate_dataset.py --data_root data --cache_root data/video_features_v1

注意：
  - 只刪「位元組完全相同」的重複檔（MD5 碰撞機率極低可忽略）
  - 同場景連拍 5 次但內容不同的影片「不會」被刪，請放心
  - 刪除記錄會存到 dedup_log.txt，可事後查閱
"""

import os
import re
import json
import hashlib
import argparse
from collections import defaultdict
from tqdm import tqdm


# =========================================================
# 與訓練腳本 yolo11_detect_person_trf.py 完全一致的 cache meta
# 若訓練腳本的 meta 有更動，請同步修改此處
# =========================================================
CACHE_META = {
    "T": 30,
    "backbone": "mbv3_small.features@224",
    "norm": "imagenet",
    "roi": {
        "type": "align", "bins": [7, 7],
        "aligned": True, "sampling": "auto", "scale": "7/224"
    },
    "sampler": "uniform_ts+segment_max+nearest_fill",
    "cached_shape": [576, 7, 7],
    "head": "DSConv+GAP",
    "version": "v1"
}

VIDEO_EXTS = (".mp4", ".mov", ".avi", ".mkv")


# =========================================================
# 工具函式
# =========================================================
def md5_of_file(path: str, chunk_bytes: int = 65536) -> str:
    """逐塊讀取，計算檔案 MD5；大檔也不會爆記憶體。"""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk_bytes), b""):
            h.update(block)
    return h.hexdigest()


def make_cache_path(video_path: str, do_flip: bool,
                    cache_root: str) -> str:
    """還原訓練腳本的 cache 路徑（與 make_cache_path() 邏輯完全相同）。"""
    key_str = (
        json.dumps(CACHE_META, sort_keys=True)
        + "|" + os.path.abspath(video_path)
        + f"|flip={int(do_flip)}"
    )
    h = hashlib.md5(key_str.encode("utf-8")).hexdigest()[:12]
    stem = os.path.splitext(os.path.basename(video_path))[0]
    fname = f"{stem}-{h}.npz"
    return os.path.join(cache_root, fname)


def collect_videos(data_root: str):
    """遞迴掃描 data_root，回傳所有影片路徑清單（已排序）。"""
    videos = []
    for root, _, files in os.walk(data_root):
        for fname in sorted(files):
            if fname.lower().endswith(VIDEO_EXTS):
                videos.append(os.path.join(root, fname))
    return sorted(videos)


def subject_of(path: str) -> str:
    """從路徑取出受試者 ID（P001~P006），找不到回傳 'unknown'。"""
    for part in path.replace("\\", "/").split("/"):
        if re.match(r"^P\d+$", part):
            return part
    return "unknown"


# =========================================================
# 主流程
# =========================================================
def main():
    ap = argparse.ArgumentParser(
        description="找出並刪除資料集中的重複影片及其快取。"
    )
    ap.add_argument("--data_root",   default="data",
                    help="影片根目錄（預設 data）")
    ap.add_argument("--cache_root",  default=os.path.join("data", "video_features_v1"),
                    help="特徵快取資料夾（預設 data/video_features_v1）")
    ap.add_argument("--dry-run",     action="store_true",
                    help="只列出要刪的清單，不實際刪除任何檔案")
    ap.add_argument("--log",         default="dedup_log.txt",
                    help="刪除記錄輸出路徑（預設 dedup_log.txt）")
    args = ap.parse_args()

    dry = args.dry_run
    mode_tag = "[DRY-RUN]" if dry else "[LIVE]"
    print(f"\n{'='*55}")
    print(f"  資料集去重工具  {mode_tag}")
    print(f"  data_root  : {args.data_root}")
    print(f"  cache_root : {args.cache_root}")
    print(f"{'='*55}\n")

    # ---- Step 1：掃描所有影片 ----
    print("[1/3] 掃描影片檔案...")
    videos = collect_videos(args.data_root)
    print(f"      找到 {len(videos)} 支影片\n")
    if not videos:
        print("找不到任何影片，請確認 --data_root 是否正確。")
        return

    # ---- Step 2：計算 MD5，按 hash 分組 ----
    print("[2/3] 計算 MD5（檔案多時需要幾分鐘，請耐心等候）...")
    hash_to_paths: dict = defaultdict(list)
    for path in tqdm(videos, unit="file", ncols=80):
        try:
            h = md5_of_file(path)
            hash_to_paths[h].append(path)
        except Exception as e:
            print(f"\n  [警告] 無法讀取 {path}: {e}")

    # 只留有重複的組
    dup_groups = {
        h: sorted(paths)
        for h, paths in hash_to_paths.items()
        if len(paths) > 1
    }
    total_dup_files = sum(len(v) - 1 for v in dup_groups.values())

    print(f"\n      共找到 {len(dup_groups)} 組重複，需刪除 {total_dup_files} 支影片")

    if not dup_groups:
        print("\n資料集中沒有完全相同的重複影片，無需刪除。")
        # 仍寫一份空 log
        with open(args.log, "w", encoding="utf-8") as f:
            f.write("no_duplicates=True\n")
        return

    # ---- Step 3：逐組處理 ----
    print(f"\n[3/3] {'預覽' if dry else '執行'}刪除...\n")

    log_lines = [
        f"dry_run={dry}",
        f"data_root={os.path.abspath(args.data_root)}",
        f"cache_root={os.path.abspath(args.cache_root)}",
        "",
    ]

    deleted_videos = 0
    deleted_caches = 0
    skipped_caches = 0   # cache 存在但非本腳本的 root，略過

    for grp_idx, (h, paths) in enumerate(sorted(dup_groups.items()), start=1):
        keep      = paths[0]          # 保留第一個（字母序最小）
        to_delete = paths[1:]

        subjects = set(subject_of(p) for p in paths)
        print(f"  組 {grp_idx:02d} [MD5 {h[:10]}...]  "
              f"受試者={sorted(subjects)}")
        print(f"    ✓ 保留: {keep}")

        log_lines.append(f"=== 組 {grp_idx:02d} [MD5={h}] ===")
        log_lines.append(f"KEEP   {keep}")

        for p in to_delete:
            print(f"    ✗ 刪除: {p}")
            log_lines.append(f"DELETE {p}")

            # 尋找對應 cache（flip=False 與 flip=True 各一份）
            for flip in (False, True):
                cp = make_cache_path(p, flip, args.cache_root)
                if os.path.exists(cp):
                    flip_tag = "flip" if flip else "orig"
                    print(f"      - cache({flip_tag}): {os.path.basename(cp)}")
                    log_lines.append(f"  CACHE {cp}")
                    if not dry:
                        try:
                            os.remove(cp)
                            deleted_caches += 1
                        except Exception as e:
                            print(f"      [警告] 刪除 cache 失敗: {e}")
                else:
                    skipped_caches += 1   # 該 cache 不存在（正常，可能未產生）

            if not dry:
                try:
                    os.remove(p)
                    deleted_videos += 1
                except Exception as e:
                    print(f"    [警告] 刪除影片失敗: {e}")
                    log_lines.append(f"  ERROR {e}")

        log_lines.append("")

    # ---- 摘要 ----
    print(f"\n{'='*55}")
    if dry:
        print(f"  [DRY-RUN] 預計刪除影片 {total_dup_files} 支")
        print(f"  實際未刪除任何檔案，確認後去掉 --dry-run 再執行")
    else:
        print(f"  完成：刪除影片 {deleted_videos} 支、"
              f"cache {deleted_caches} 份")
    print(f"{'='*55}\n")

    # ---- 寫 log ----
    log_lines += [
        "",
        f"=== 摘要 ===",
        f"dry_run         = {dry}",
        f"dup_groups      = {len(dup_groups)}",
        f"deleted_videos  = {deleted_videos}",
        f"deleted_caches  = {deleted_caches}",
        f"skipped_caches  = {skipped_caches} (cache 不存在，無需刪除)",
    ]
    with open(args.log, "w", encoding="utf-8") as f:
        f.write("\n".join(log_lines) + "\n")
    print(f"  記錄已寫入：{args.log}")


if __name__ == "__main__":
    main()
